In [58]:
import sys
sys.path.append('../')
import pandas as pd
import datetime as dt
from plotting import CandlePlot
import plotly.graph_objects as go
from infrastructure.instrument_collection import instrumentCollection as ic

In [59]:
pair = 'GBP_JPY'
granularity = 'H4'
df = pd.read_pickle(f'../data/{pair}_{granularity}.pkl')
MA_LIST = [10, 20, 50, 100, 200]

In [60]:

df_ma = df[['time','mid_o','mid_h','mid_l','mid_c']].copy()

In [61]:
df_ma.tail()

,time,mid_o,mid_h,mid_l,mid_c
3995,2025-01-26 22:00:00+00:00,194.226,194.361,193.474,193.563
3996,2025-01-27 02:00:00+00:00,193.569,194.316,193.471,194.252
3997,2025-01-27 06:00:00+00:00,194.252,194.654,192.606,193.044
3998,2025-01-27 10:00:00+00:00,193.044,193.152,192.158,192.805
3999,2025-01-27 14:00:00+00:00,192.806,193.078,192.174,192.482


In [62]:
# Note that the first 9 rows will have NaN values in the MA_10 column
# Similarly, the first 19 rows will have NaN values in the MA_20 column.
# We need to pay attention to this while doing analysis, and backtesting.

for ma in MA_LIST:
    df_ma[f'MA_{ma}'] = df_ma['mid_c'].rolling(window=ma).mean()


df_ma.dropna(inplace = True)
df_ma.reset_index(drop=True, inplace=True)
df_ma.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_10,MA_20,MA_50,MA_100,MA_200
0,2022-08-17 05:00:00+00:00,162.583,163.392,162.482,163.174,161.8321,161.71250,162.32144,162.62093,163.256840
1,2022-08-17 09:00:00+00:00,163.174,163.368,162.896,162.974,162.0605,161.74000,162.34340,162.60435,163.253090
2,2022-08-17 13:00:00+00:00,162.976,163.570,162.792,163.032,162.3270,161.77840,162.37156,162.58949,163.248335
3,2022-08-17 17:00:00+00:00,163.030,163.224,162.622,162.752,162.5194,161.82035,162.38692,162.56981,163.249945
4,2022-08-17 21:00:00+00:00,162.701,162.877,162.430,162.468,162.6436,161.84195,162.37192,162.55127,163.247515


In [63]:
df_plot = df_ma.iloc[:500]
df_plot.shape
print(df_plot.columns)

cp = CandlePlot(df_plot)
traces = [f"MA_{x}" for x in MA_LIST]
cp.show_plot(line_traces=traces)

Index(['time', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'MA_10', 'MA_20', 'MA_50',
       'MA_100', 'MA_200'],
      dtype='object')


In [64]:
MA_S = "MA_50"
MA_L = "MA_200"
BUY = 1
SELL = -1
NONE = 0


In [65]:
df_an = df_ma[['time','mid_o','mid_h','mid_l','mid_c',MA_S,MA_L]].copy()
df_an.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_50,MA_200
0,2022-08-17 05:00:00+00:00,162.583,163.392,162.482,163.174,162.32144,163.256840
1,2022-08-17 09:00:00+00:00,163.174,163.368,162.896,162.974,162.34340,163.253090
2,2022-08-17 13:00:00+00:00,162.976,163.570,162.792,163.032,162.37156,163.248335
3,2022-08-17 17:00:00+00:00,163.030,163.224,162.622,162.752,162.38692,163.249945
4,2022-08-17 21:00:00+00:00,162.701,162.877,162.430,162.468,162.37192,163.247515


In [66]:
df_an['DELTA'] = df_an.MA_50 - df_an.MA_200
df_an.head(50)

,time,mid_o,mid_h,mid_l,mid_c,MA_50,MA_200,DELTA
0,2022-08-17 05:00:00+00:00,162.583,163.392,162.482,163.174,162.32144,163.256840,-0.935400
1,2022-08-17 09:00:00+00:00,163.174,163.368,162.896,162.974,162.34340,163.253090,-0.909690
2,2022-08-17 13:00:00+00:00,162.976,163.570,162.792,163.032,162.37156,163.248335,-0.876775
3,2022-08-17 17:00:00+00:00,163.030,163.224,162.622,162.752,162.38692,163.249945,-0.863025
4,2022-08-17 21:00:00+00:00,162.701,162.877,162.430,162.468,162.37192,163.247515,-0.875595
5,2022-08-18 01:00:00+00:00,162.468,162.690,162.370,162.630,162.36518,163.243080,-0.877900
6,2022-08-18 05:00:00+00:00,162.628,163.083,162.284,162.894,162.35942,163.240840,-0.881420
7,2022-08-18 09:00:00+00:00,162.896,163.246,162.430,162.584,162.34426,163.236750,-0.892490
8,2022-08-18 13:00:00+00:00,162.589,162.637,161.684,161.915,162.31638,163.226055,-0.909675
9,2022-08-18 17:00:00+00:00,161.914,162.277,161.910,162.136,162.29006,163.213900,-0.923840


In [67]:
# We need to look for delta sign changes to identify buy/sell signals
df_an['DELTA_PREV'] = df_an['DELTA'].shift(1)


In [68]:
def is_trade(row):
    if row['DELTA'] > 0 and row['DELTA_PREV'] < 0:
        return BUY
    elif row['DELTA'] < 0 and row['DELTA_PREV'] > 0:
        return SELL
    return NONE

In [69]:
df_an['TRADE'] = df_an.apply(is_trade, axis=1)

In [70]:
# df_an.head(50)
df_an.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_50,MA_200,DELTA,DELTA_PREV,TRADE
0,2022-08-17 05:00:00+00:00,162.583,163.392,162.482,163.174,162.32144,163.256840,-0.935400,NaN,0
1,2022-08-17 09:00:00+00:00,163.174,163.368,162.896,162.974,162.34340,163.253090,-0.909690,-0.935400,0
2,2022-08-17 13:00:00+00:00,162.976,163.570,162.792,163.032,162.37156,163.248335,-0.876775,-0.909690,0
3,2022-08-17 17:00:00+00:00,163.030,163.224,162.622,162.752,162.38692,163.249945,-0.863025,-0.876775,0
4,2022-08-17 21:00:00+00:00,162.701,162.877,162.430,162.468,162.37192,163.247515,-0.875595,-0.863025,0


In [71]:

df_trades = df_an[df_an['TRADE'] != NONE].copy()
print(df_trades.shape)
df_trades.head()

(22, 10)


,time,mid_o,mid_h,mid_l,mid_c,MA_50,MA_200,DELTA,DELTA_PREV,TRADE
94,2022-09-07 21:00:00+00:00,165.785,166.325,165.761,166.083,162.52420,162.482745,0.041455,-0.039300,1
166,2022-09-25 21:00:00+00:00,155.138,155.640,149.333,149.497,162.73530,162.746095,-0.010795,0.256970,-1
235,2022-10-11 09:00:00+00:00,160.909,161.766,160.778,161.505,162.13024,162.123840,0.006400,-0.101630,1
388,2022-11-15 22:00:00+00:00,165.286,165.706,164.708,165.493,166.24186,166.275035,-0.033175,0.000815,-1
513,2022-12-14 18:00:00+00:00,167.771,168.395,167.361,168.376,167.11960,167.093855,0.025745,-0.052325,1


In [72]:
cp = CandlePlot(df_an.iloc[:65])
cp.show_plot(line_traces=[MA_S,MA_L])

In [73]:
# df.groupby(stuff...).sum(numeric_only=True) will sum only the numeric columns

In [74]:
ic.LoadInstruments("../data")

In [75]:
ic.instruments_dict[pair]

{'name': 'GBP_JPY', 'ins_type': 'CURRENCY', 'displayName': 'GBP/JPY', 'pipLocation': 0.01, 'tradeUnitsPrecision': 0, 'marginRate': 0.05}

In [76]:
ins_data = ic.instruments_dict[pair]

In [77]:
df_trades.shape
# Note that every trade will have a pip difference, this will cause some loss in the backtest

(22, 10)

In [78]:
df_trades['DIFF'] = df_trades['mid_c'].diff().shift(-1)
df_trades.fillna(0, inplace=True)

In [79]:
df_trades['GAIN'] = df_trades['DIFF'] / ins_data.pipLocation
df_trades['GAIN'] = df_trades['GAIN'] * df_trades['TRADE']

In [80]:
df_trades.GAIN.sum()

np.float64(-2469.0999999999917)

In [81]:
df_trades['GAIN_C'] = df_trades['GAIN'].cumsum()
df_trades.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_50,MA_200,DELTA,DELTA_PREV,TRADE,DIFF,GAIN,GAIN_C
94,2022-09-07 21:00:00+00:00,165.785,166.325,165.761,166.083,162.52420,162.482745,0.041455,-0.039300,1,-16.586,-1658.6,-1658.6
166,2022-09-25 21:00:00+00:00,155.138,155.640,149.333,149.497,162.73530,162.746095,-0.010795,0.256970,-1,12.008,-1200.8,-2859.4
235,2022-10-11 09:00:00+00:00,160.909,161.766,160.778,161.505,162.13024,162.123840,0.006400,-0.101630,1,3.988,398.8,-2460.6
388,2022-11-15 22:00:00+00:00,165.286,165.706,164.708,165.493,166.24186,166.275035,-0.033175,0.000815,-1,2.883,-288.3,-2748.9
513,2022-12-14 18:00:00+00:00,167.771,168.395,167.361,168.376,167.11960,167.093855,0.025745,-0.052325,1,-7.539,-753.9,-3502.8


In [82]:

cp = CandlePlot(df_trades,candles= False)
cp.show_plot(line_traces=['GAIN_C'])